<div class="alert alert-info" style="background-color:#001b79; color:white; padding:0px 10px; border-radius:5px;"><h2 style='margin:10px 5px'>Recommendation Engine UBCF CBCF </h2>
</div>

> Is a tool, that allows algorithm developers predict what user may or may not like  among a list of given items.

> Help users discover products or content that we may not come across otherwise

### RE works ideally 2 ways:		
####	1. User Base filtering (UB)/User Based Collaborative Filtering (UBCF):	
		Building a model based on users past behaviour as well as similar decision made by other users.
		This model is used to predict items that the users may have an interset in.
		But in this tend to recommend only popular items and fail to recommend new or unique taste one
		
		
		
####	2. Content- Based Filtering(CB)	
		Utilizes series of discrete characterestics of an item in order to recommend additional items to users with similar properties.
		Here items characterstics are identified , similar characterstics other items will be recommended.
        Here unique taste one will be recommended but never recommend only popular items outside users content profile
		
		
####	3. Hybrid Recommendation system:	
		This is combined feature of above both filtering, this gives better result




<div class="alert alert-info" style="background-color:#001b32; color:white; padding:0px 10px; border-radius:5px;"><h2 style='margin:10px 5px'>Problem  Statement </h2>
</div>

## Modiule 11 :  Association Rules Mining and Recommendation Systems

### Domain –Retail

#### focus –Optimize Book RENT

#### Business challenge/requirement
- BookRent is the largest online and offline book rental chain in India. 
- TheCompany charges a fixed fee per month plus rental per book. 
- So,the company makes more money when user rent more books. 
- You as an ML expert have to model recommendation engine so that user gets recommendation of books based on the behaviorof similar users. 
- This will ensure that users are renting books based on their individual taste.
- Company is still unprofitable and is looking to improve both revenue and profit.

#### Key issues
- As of now lot users return book and do not take new rental.  
- Right recommendation will entice user to rent more books

#### Considerations
- NONE

#### Data volume
- Approx 1 M  records –file BX-Book-Ratings.csv and 2 more. But only 10K records will be used

#### Fields in Data
    •user_id: Unique Id of the User
    •isbn: International Standard Book Number is a unique numeric commercial book identifier
    •rating: rating given by user

#### Additional information-NA

#### Business benefits
- Increase in both top line and bottom line as more rentals per user means more revenue and more profit



##  1. Import all Librariy

#### We have 3  file 1) BX-Users.csv 2)BX-Books.csv 3) BX-Book-Ratings.csv

> We are Creating Cosine similarity matrix for users based recommendation

In [60]:
import pandas as pd
import numpy as np

#got "UnicodeDecodeError: 'utf-8' codec can't decode byte 0xed in position 12: unexpected end of data", so added encoding = "ISO-8859-1" 

#df = pd.read_csv(r"..\Dataset\Module11\BX_Book_Ratings.csv",encoding = "ISO-8859-1")

df_user = pd.read_csv(r"..\Dataset\BX-Users.csv",encoding = "latin-1")
df_user.head()

C:\Users\prabh\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3020: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,user_id,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


We will ignore this data, so need to clean up NaN values

In [61]:
#Read the books Data and explore
df_books = pd.read_csv(r"..\Dataset\BX-Books.csv",encoding = "latin-1")
df_books.head()

C:\Users\prabh\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3020: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,isbn,book_title,book_author,year_of_publication,publisher
0,195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,2005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,60973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


In [62]:
## Now read the data where ratings by users. We will read only first 10K rows otherwise Out Of Memory error can occur

df = pd.read_csv(r"..\Dataset\BX-Book-Ratings.csv" ,encoding = "latin-1",nrows=10000)
df.head()

,user_id,isbn,rating
0,276725,034545104X,0
1,276726,155061224,5
2,276727,446520802,0
3,276729,052165615X,3
4,276729,521795028,6


## 2. Data Exploration

In [66]:
#Numpy instalation prob, describe and info not working
df.info()

Merge the dataframes df and df_books. For all practical purpose User Master Data is not required. So we will ignore dataframe df_user

### 2.1 Data Merging

In [67]:
df = pd.merge(df,df_books,on='isbn')
df.head()

,user_id,isbn,rating,book_title,book_author,year_of_publication,publisher
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books
1,276726,155061224,5,Rites of Passage,Judith Rae,2001,Heinle
2,276727,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books
3,278418,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books
4,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press


Now let's take a quick look at the number of unique users and books.

In [68]:
n_users = df.user_id.nunique()
n_books = df.isbn.nunique()

# str converts to string  to conacateneate(+) with string
print('Num. of Users: '+ str(n_users))
print('Num of Books: '+str(n_books))

Num. of Users: 828
Num of Books: 8051


### 2.2 Converting ISBN and USERID to unique ordered numeric no for that create function


> Here below Function Convert isbn to  unique numeric numbers in ordered way

In [69]:
isbn_list = df.isbn.unique()
print(" Length of isbn List:", len(isbn_list))
print(isbn_list)
def get_isbn_numeric_id(isbn):
    #print ("  isbn is:" , isbn)
    #These provides index no/row no for matching isbn
    itemindex = np.where(isbn_list==isbn)
    return itemindex[0][0]

 Length of isbn List: 8051
['034545104X' '155061224' '446520802' ... '425098834' '425163407'
 '425164403']



> Here below Function Convert user_id to  unique numeric numbers in ordered way

In [70]:
userid_list = df.user_id.unique()
print(" Length of user_id List:", len(userid_list))
print(userid_list[:10])
def get_user_id_numeric_id(user_id):
    #print ("  isbn is:" , isbn)
    #These provides index no/unique no to for matching user_id
    itemindex = np.where(userid_list==user_id)
    return itemindex[0][0]

 Length of user_id List: 828
[276725 276726 276727 278418 276729 276733 276744 276746 277427 278026]


### 2.3 Converting ISBN and USERID to unique ordered numeric no by using above user created function
Converting both user_id and isbn to ordered list i.e. from 0...n-1

#### User id and isbn are not continous no, in this handling we are creating matrix based on this, so it should be continous values

In [71]:
#New col user_id_order
df['user_id_order'] = df['user_id'].apply(get_user_id_numeric_id)
df

,user_id,isbn,rating,book_title,book_author,year_of_publication,publisher,user_id_order
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,0
1,276726,155061224,5,Rites of Passage,Judith Rae,2001,Heinle,1
2,276727,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,2
3,278418,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,3
4,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,4
5,276729,521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,4
6,276733,2080674722,0,Les Particules Elementaires,Michel Houellebecq,1998,Flammarion,5
7,276744,038550120X,7,A Painted House,JOHN GRISHAM,2001,Doubleday,6
8,278418,038550120X,0,A Painted House,JOHN GRISHAM,2001,Doubleday,3
9,276746,425115801,0,Lightning,Dean R. Koontz,1996,Berkley Publishing Group,7


In [72]:
#New col isbn_id
df['isbn_id'] = df['isbn'].apply(get_isbn_numeric_id)
df.head()

,user_id,isbn,rating,book_title,book_author,year_of_publication,publisher,user_id_order,isbn_id
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,0,0
1,276726,155061224,5,Rites of Passage,Judith Rae,2001,Heinle,1,1
2,276727,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,2,2
3,278418,446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,3,2
4,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,4,3


Re-index columns to build matrix later on

In [73]:
new_col_order = ['user_id_order', 'isbn_id', 'rating', 'book_title', 'book_author','year_of_publication','publisher','isbn','user_id']
# Here re_index will re arrange or sort the rows based on the values in columns= new_col_order 
df = df.reindex(columns= new_col_order)
df.head()

,user_id_order,isbn_id,rating,book_title,book_author,year_of_publication,publisher,isbn,user_id
0,0,0,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,034545104X,276725
1,1,1,5,Rites of Passage,Judith Rae,2001,Heinle,155061224,276726
2,2,2,0,The Notebook,Nicholas Sparks,1996,Warner Books,446520802,276727
3,3,2,0,The Notebook,Nicholas Sparks,1996,Warner Books,446520802,278418
4,4,3,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,052165615X,276729


## 3. Building Machine Learning Recommendation engine - Collaborative Filtering

### 3.1 Train Test Split

- Recommendation Systems are difficult to evaluate, but we will still show you how to evaluate them. In order to do this, we'll split our data into two sets. However, we won't do our classic X_train,X_test,y_train,y_test split. Instead we can actually just segement the data into two sets of data:

In [74]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size=0.30)

train_data.head()

,user_id_order,isbn_id,rating,book_title,book_author,year_of_publication,publisher,isbn,user_id
5798,3,5182,0,"Ty Cobb, the Greatest (Putnam Sports Shelf)",Robert Rubin,1977,Putnam Pub Group (L),039961110X,278418
3005,162,2499,0,Great Deliverance (Thomas Lynley and Barbara H...,Elizabeth George,1988,Bantam Dell Pub Group,553052446,277928
693,157,580,8,Mutant Message: Down Under,Marlo Morgan,1992,Bookpeople,1883473004,277031
7583,3,6951,0,Lord Ravenscar's Revenge,Baraba Cartland,1984,John Curley &amp; Assoc,860096041,278418
3866,83,3275,0,Possessing the Secret of Joy,Alice Walker,1993,Pocket,671789422,278194


### 3.2  Approach --  We will use Memory-Based Collaborative Filtering


- Memory-Based Collaborative Filtering approaches can be divided into two main sections: **user-item filtering** and **item-item filtering**. 

- A *user-item filtering* will take a particular user, find users that are similar to that user based on similarity of ratings, and recommend items that those similar users liked. 

- In contrast, *item-item filtering* will take an item, find users who liked that item, and find other items that those users or similar users also liked. It takes items and outputs other items as recommendations. 
        - Based on the content of books it recommends similar item

* *Item-Item Collaborative Filtering*: “Users who liked this item also liked …”  Something like content based
* *User-Item Collaborative Filtering*: “Users who are similar to you also liked …”

> In both cases, you create a user-book matrix which built from the entire dataset.

> Since we have split the data into testing and training we will need to create two ``[828 x 8051]`` matrices (all users by all books). This is going to be  a very large matrix

> The training matrix contains 70% of the ratings and the testing matrix contains 30% of the ratings.  

### 3.2.1 Create (no of unique user) * (no of unique books)  matrix

In [75]:
#Create two user-book matrices, one for training and another for testing
train_data_matrix = np.zeros((n_users, n_books))

# itertuples: returns a named tuple with indexvalue and colvalues
#  line is : its tuple of index value and all col value => user_id_order	isbn_id	rating	book_title	book_author	year_of_publication	publisher	isbn	user_id
#    #Pandas(Index=2108, user_id_order=352, isbn_id=1727, rating=0, book_title="A Thousand Acres (Ballantine Reader's Circle)", book_author='JANE SMILEY', year_of_publication='1992', publisher='Ballantine Books', isbn='449907481', user_id=277554)

# Here line[3]  is 4th value in tuple which is rating
# line[1]-1 provide user id value , line[2]-1 book_id value. user_id and book_id are continous value in line datset
# train_data_matrix[0,0] --> 1st row 1st col  position in matrix
# train_data_matrix[0,1] --> 1st row 2nd col  position in matrix

for line in train_data.itertuples():
    #print(line)
    #Pandas(Index=2108, user_id_order=352, isbn_id=1727, rating=0, book_title="A Thousand Acres (Ballantine Reader's Circle)", book_author='JANE SMILEY', year_of_publication='1992', publisher='Ballantine Books', isbn='449907481', user_id=277554)
    # This assign rating value in each matrix position
    train_data_matrix[line[1]-1, line[2]-1] = line[3]  

    
train_data_matrix

#Below 1 st row 1st value is 5. means, user_id=1 provided  Rating =5  for n_books=1 
#Below 1st row 2nd value is 3. means, user_id=1 provided  rating =3  for n_books=2
#Below 2nd row 1st value is 4. means, user_id=2 provided  rating =4  for n_books=1

array([[5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [76]:
# do the same for testing data

test_data_matrix = np.zeros((n_users, n_books))

for line in test_data.itertuples():
     test_data_matrix[line[1]-1, line[2]-1] = line[3]

### 3.2.2 Apply pairwise_distances function with cosine similarity
You can use the pairwise_distances function from sklearn to calculate the cosine similarity. Note, the output will range from 0 to 1 since the ratings are all positive

In [77]:
from sklearn.metrics.pairwise import pairwise_distances

user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')

C:\Users\prabh\Anaconda3\lib\site-packages\sklearn\metrics\pairwise.py:54: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dtype = np.float
C:\Users\prabh\Anaconda3\lib\site-packages\sklearn\metrics\pairwise.py:54: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dtype = np.float


In [78]:
user_similarity

array([[0., 1., 1., ..., 1., 1., 1.],
       [1., 0., 1., ..., 1., 1., 1.],
       [1., 1., 0., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 0., 1., 1.],
       [1., 1., 1., ..., 1., 0., 1.],
       [1., 1., 1., ..., 1., 1., 0.]])

In [79]:
item_similarity

array([[0., 1., 1., ..., 1., 1., 1.],
       [1., 0., 1., ..., 1., 1., 1.],
       [1., 1., 0., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 0., 1., 1.],
       [1., 1., 1., ..., 1., 0., 1.],
       [1., 1., 1., ..., 1., 1., 0.]])

### 3.2.3 Make Prediction

    - Next step is to make predictions

In [80]:
def predict(ratings, similarity, type='user'):
    
    if type == 'user':
        mean_user_rating = ratings.mean(axis=1)
        #You use np.newaxis so that mean_user_rating has same format as ratings
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) 
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])     
    return pred

In [81]:
item_prediction = predict(train_data_matrix, item_similarity, type='item')
user_prediction = predict(train_data_matrix, user_similarity, type='user')

In [82]:
item_prediction

array([[0.        , 0.00062112, 0.0006212 , ..., 0.00062112, 0.00062112,
        0.00062112],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.0557764 , 0.0557764 , 0.05578333, ..., 0.0557764 , 0.0557764 ,
        0.0557764 ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

## 4. Model Evaluation 
 - There are many evaluation metrics but one of the most popular metric used to evaluate accuracy of predicted ratings is *Root Mean Squared Error (RMSE)*.

> ince you only want to consider predicted ratings that are in the test dataset, you filter out all other elements in the prediction matrix with `prediction[ground_truth.nonzero()]`.

In [83]:
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten() 
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return sqrt(mean_squared_error(prediction, ground_truth))

In [84]:
print('User-based CF RMSE: ' + str(rmse(user_prediction, test_data_matrix)))
print('Item-based CF RMSE: ' + str(rmse(item_prediction, test_data_matrix)))

User-based CF RMSE: 7.736022771714497
Item-based CF RMSE: 7.735393839769416


## 5. Conclusion

> Both the approach yield almost same result

> How to understand predicted value is not clear yet. yet to know more